# Grading INFO 4617 from Canvas

This notebook grades the course's recurring work by reading from and posting to Canvas:
**attendance**, **Textbook Revisions** (weeks 2–9), the **portfolio checkpoints** (weeks 11–14),
and **Notebook Labs**. The rules are in [`docs/grading.md`](../docs/grading.md).

Each section scores only the **plain case**: a submission that meets the assignment as written, with
nothing to decide. Everything else (a late lab, a missing step, a note from a personal email, a
link to someone else's issue) is printed in its own cell for you to decide. Those rows are left out
of the section's table, so you enter them in Canvas yourself. Nothing here applies a penalty, an
exception, or a comment.

Run it top to bottom. Each section saves its table of plain rows as a CSV in `GRADING_DIR`. The
last section posts one table to Canvas, but only with `APPLY = True` and after you type how many
rows you're posting. It then reads every score back from Canvas.

**Clear the outputs before you commit this notebook**: they show students' names and scores, and
the repository is public. Student data lives only in `GRADING_DIR`, outside the repository.

## Setup

The libraries, and `course_spec.py` from this folder for the semester's dates.

In [ ]:
import json
import os
import re
import subprocess
import sys
from collections import Counter, defaultdict
from datetime import datetime, timedelta
from pathlib import Path

import pandas as pd
import requests
from bs4 import BeautifulSoup

sys.path.insert(0, ".")
import course_spec as spec

Set `CANVAS_TOKEN` in your environment (Canvas → Account → Settings → New Access Token).
`GITHUB_TOKEN` is optional: without it GitHub allows 60 requests an hour, which isn't enough for
a week of revisions. A read-only fine-grained token with no repository access is enough.

In [ ]:
APPLY = False     # True only when you are ready to post a table in the last section

CANVAS = f"https://canvas.colorado.edu/api/v1/courses/{spec.NEW_COURSE_ID}"
GRADING_DIR = Path(os.environ.get("GRADING_DIR", Path.home() / "info4617-grading"))
GRADING_DIR.mkdir(parents=True, exist_ok=True)

canvas = requests.Session()
canvas.headers["Authorization"] = "Bearer " + os.environ["CANVAS_TOKEN"].strip()
github = requests.Session()
github.headers["Accept"] = "application/vnd.github+json"
if os.environ.get("GITHUB_TOKEN"):
    github.headers["Authorization"] = "Bearer " + os.environ["GITHUB_TOKEN"].strip()

# the class repositories a revision can be in (docs/grading.md)
CLASS_REPOS = {"cuinfoscience/web-data-science-book", "cuinfoscience/info4617-fall2026", "cuinfoscience/info-missing-manual"}
# the columns of every table this notebook saves and posts
COLUMNS = ["kind", "week", "assignment_id", "user_id", "name", "score", "rubric", "notes", "url"]

print("Student data goes to", GRADING_DIR)
print("Posting is", "ON" if APPLY else "off (dry run)")

## The roster and the assignments

Every later section uses these two tables. Canvas returns long lists a page at a time, so each
request follows the `next` link until there isn't one.

In [ ]:
url, params, users = f"{CANVAS}/users", {"enrollment_type[]": "student", "per_page": 100}, []
while url:
    r = canvas.get(url, params=params)
    r.raise_for_status()
    users += r.json()
    url, params = r.links.get("next", {}).get("url"), None

roster = pd.DataFrame({
    "user_id": [u["id"] for u in users],
    "name": [u["name"] for u in users],
    "identikey": [(u.get("login_id") or "").lower() for u in users],
}).set_index("user_id")
print(len(roster), "students")
roster.head()

In [ ]:
url, params, groups = f"{CANVAS}/assignment_groups", {"include[]": "assignments", "per_page": 100}, []
while url:
    r = canvas.get(url, params=params)
    r.raise_for_status()
    groups += r.json()
    url, params = r.links.get("next", {}).get("url"), None

rows = []
for g in groups:
    for a in g["assignments"]:
        m = re.match(r"Week (\d+)", a["name"])
        rows.append({"assignment_id": a["id"], "name": a["name"], "group": g["name"],
                     "week": int(m.group(1)) if m else None, "points": a["points_possible"],
                     "due_at": a.get("due_at"), "published": a["published"],
                     "has_rubric": bool(a.get("rubric"))})
assignments = pd.DataFrame(rows)
print(len(assignments), "assignments")
assignments.groupby("group")["assignment_id"].count()

## What's left to grade

Submissions in published assignments that have no score, or were resubmitted after they were
graded.

In [ ]:
left = []
for _, a in assignments[assignments["published"] & assignments["group"].isin(["Notebook Labs", "Textbook Revisions", "Final Project"])].iterrows():
    url, params, subs = f"{CANVAS}/assignments/{a['assignment_id']}/submissions", {"per_page": 100}, []
    while url:
        r = canvas.get(url, params=params)
        r.raise_for_status()
        subs += r.json()
        url, params = r.links.get("next", {}).get("url"), None
    for s in subs:
        if s["user_id"] in roster.index and s.get("submitted_at"):
            left.append({"assignment": a["name"], "name": roster.loc[s["user_id"], "name"], "score": s.get("score"),
                         "submitted_at": s["submitted_at"][:16], "graded_at": (s.get("graded_at") or "")[:16]})
left = pd.DataFrame(left, columns=["assignment", "name", "score", "submitted_at", "graded_at"])
left = left[left["score"].isna() | ((left["graded_at"] != "") & (left["submitted_at"] > left["graded_at"]))]
print(len(left), "submissions with no score or resubmitted after grading")
left

## Attendance

Attendance is the **daily note**, the Google Form students fill in at the end of class. Download
its responses as a CSV (Responses → Download) to `GRADING_DIR / "daily_note.csv"`. It stays out
of the repository.

A note is **plain** when it comes from a roster student's `identikey@colorado.edu` address, was
sent on one of the week's meeting days, and gives that day's number. A student's score is the
number of classes they sent a plain note for. Any other note is printed below for you to decide.

In [ ]:
WEEK = 11

note = pd.read_csv(GRADING_DIR / "daily_note.csv", dtype=str).fillna("")
note.columns = note.columns.str.strip()
print(len(note), "responses; columns:", list(note.columns))
note[["Timestamp", "Email Address", "Today is", "What was today's number?"]].head()

Parse the four columns the grading reads. The form writes `Timestamp` as month/day/year, in
Mountain time. A timestamp that doesn't parse is printed and dropped.

In [ ]:
note["sent"] = pd.to_datetime(note["Timestamp"].str.strip(), format="%m/%d/%Y %H:%M:%S", errors="coerce")
note["today_is"] = pd.to_datetime(note["Today is"].str.strip(), format="%m/%d/%Y", errors="coerce").dt.date
note["email"] = note["Email Address"].str.strip().str.lower()
note["number"] = note["What was today's number?"].str.strip()
print(note["sent"].isna().sum(), "timestamps that don't parse:", note.loc[note["sent"].isna(), "Timestamp"].tolist())
note = note.dropna(subset=["sent"])
note["sent_on"] = note["sent"].dt.date
print("Notes from", note["sent_on"].min(), "to", note["sent_on"].max())
note[["sent", "email", "today_is", "number"]].head()

Keep the notes sent this week, Monday to Sunday, and count them by the day they were sent.

In [ ]:
monday = spec.week_monday(WEEK)
meeting_days = spec.meeting_days(WEEK)
week_notes = note[(note["sent_on"] >= monday) & (note["sent_on"] <= monday + timedelta(days=6))].copy()
print("Meeting days:", ", ".join(f"{d:%a %b %d}" for d in meeting_days))
print(len(week_notes), "notes sent this week")
week_notes["sent_on"].value_counts().sort_index()

Each day's number is the answer most of that day's notes give. Check it against the number you
announced.

In [ ]:
rows = []
for d in meeting_days:
    answers = week_notes.loc[(week_notes["sent_on"] == d) & (week_notes["number"] != ""), "number"].value_counts()
    rows.append({"day": d, "number": answers.index[0] if len(answers) else "",
                 "notes": int((week_notes["sent_on"] == d).sum()), "giving it": int(answers.iloc[0]) if len(answers) else 0})
day_numbers = pd.DataFrame(rows)
day_numbers

A class is held when its meeting day has notes. A meeting day with none is printed: the class was
cancelled, or the form wasn't opened. The assignment should be worth one point per class held;
if it isn't, change its points in Canvas before you post, while it has no grades.

In [ ]:
held = day_numbers[day_numbers["notes"] > 0]
print("Meeting days with no notes:", ", ".join(f"{d:%a %b %d}" for d in day_numbers.loc[day_numbers["notes"] == 0, "day"]) or "none")
a = assignments[assignments["name"] == f"Week {WEEK:02d} Attendance"].iloc[0]
print(f"{a['name']} is worth {a['points']:g} points in Canvas, and {len(held)} classes were held")

Match each note to a student by its address: the part before `@colorado.edu`, when that is an
identikey on the roster.

In [ ]:
week_notes["identikey"] = week_notes["email"].str.replace("@colorado.edu", "", regex=False)
week_notes.loc[~week_notes["email"].str.endswith("@colorado.edu") | ~week_notes["identikey"].isin(roster["identikey"]), "identikey"] = ""
print((week_notes["identikey"] != "").sum(), "notes matched to a student;", (week_notes["identikey"] == "").sum(), "not matched")

Mark the plain notes: matched, sent on a meeting day, with that day's number.

In [ ]:
number_of = dict(zip(held["day"], held["number"]))
week_notes["plain"] = (week_notes["identikey"] != "") & (week_notes["number"] == week_notes["sent_on"].map(number_of))
week_notes["plain"].value_counts()

### Notes for you to decide

**Addresses that aren't a roster identikey**, such as a personal or school email. The notebook
can't tell whose they are. Their students' rows are posted from their identikey notes alone, so
raise those scores in Canvas for the notes you count.

In [ ]:
unmatched = week_notes[week_notes["identikey"] == ""]
print(len(unmatched), "notes from", unmatched["email"].nunique(), "addresses")
unmatched[["sent", "email", "today_is", "number"]].sort_values(["email", "sent"])

**Notes from roster students that aren't plain**: sent on a day the class doesn't meet, or with a
number that isn't that day's. `matches` names any class whose number it is. A note sent a day or
more after its class usually gives that class's number. A note for this week's Friday that was
sent next week shows up when you grade next week.

These students' rows are left out of the table, so enter their scores in Canvas yourself.

In [ ]:
numbers_so_far = defaultdict(list)
for w in range(1, WEEK + 1):
    for d in spec.meeting_days(w):
        answers = note.loc[(note["sent_on"] == d) & (note["number"] != ""), "number"].value_counts()
        if len(answers):
            numbers_so_far[answers.index[0]].append(f"{d:%b %d}")

odd = week_notes[(week_notes["identikey"] != "") & ~week_notes["plain"]].copy()
odd["meeting_day"] = odd["sent_on"].isin(meeting_days)
odd["matches"] = [", ".join(numbers_so_far.get(n, [])) for n in odd["number"]]
print(len(odd), "notes from", odd["identikey"].nunique(), "students")
odd[["sent", "identikey", "today_is", "number", "meeting_day", "matches"]].sort_values("sent")

### The plain table

One row per student, except those with a note printed in the last cell. The score is the number
of classes they sent a plain note for; a note sent twice counts once.

In [ ]:
plain = week_notes[week_notes["plain"]]
print(len(plain) - len(plain.drop_duplicates(["identikey", "sent_on"])), "repeated notes counted once")
attended = plain.groupby("identikey")["sent_on"].nunique()

rows = []
for user_id, st in roster[~roster["identikey"].isin(odd["identikey"])].iterrows():
    n = int(attended.get(st["identikey"], 0))
    rows.append({"kind": "attendance", "week": WEEK, "assignment_id": a["assignment_id"], "user_id": user_id,
                 "name": st["name"], "score": n, "rubric": "", "notes": f"{n} of {len(held)} classes", "url": ""})
attendance = pd.DataFrame(rows, columns=COLUMNS)
path = GRADING_DIR / f"attendance-w{WEEK:02d}.csv"
attendance.to_csv(path, index=False)
print(len(attendance), "rows saved to", path)
print("Left for you:", ", ".join(sorted(roster.loc[roster["identikey"].isin(odd["identikey"]), "name"])) or "nobody")
attendance["score"].value_counts().sort_index()

## Textbook Revisions, weeks 2–9

A revision is **plain** when its link is an issue, pull request, comment, review, commit, or
compare page that the student's own GitHub account made, in a class repository or the student's
own fork, and it wasn't also submitted for another week. It earns 1. No submission earns 0. Every
other link is printed below for you to decide.

First, the submissions for all eight weeks.

In [ ]:
revision_assignments = assignments[(assignments["group"] == "Textbook Revisions") & assignments["week"].between(2, 9)]
rows = []
for _, a in revision_assignments.iterrows():
    url, params = f"{CANVAS}/assignments/{a['assignment_id']}/submissions", {"per_page": 100}
    while url:
        r = canvas.get(url, params=params)
        r.raise_for_status()
        for s in r.json():
            if s["user_id"] in roster.index:
                rows.append({"week": int(a["week"]), "assignment_id": a["assignment_id"], "user_id": s["user_id"],
                             "name": roster.loc[s["user_id"], "name"], "url": (s.get("url") or "").strip(),
                             "submitted": s["workflow_state"] != "unsubmitted" and bool((s.get("url") or "").strip())})
        url, params = r.links.get("next", {}).get("url"), None
revs = pd.DataFrame(rows)
print(len(revs), "rows;", revs["submitted"].sum(), "with a link")
revs.groupby("week")["submitted"].sum()

Read each link: its repository, and whether it's an issue, a pull request, a comment or review on
one, a commit, a compare page, or something else.

In [ ]:
links = {}
for link in sorted(set(revs["url"]) - {""}):
    d = {"type": "not github", "owner": "", "repo": "", "number": None, "comment_id": None, "review_id": None, "sha": None, "compare": None}
    m = re.match(r"https?://github\.com/([^/]+)/([^/#?]+)(?:/(issues|pull|commit|compare|blob|tree)/([^#?]+))?(?:\?[^#]*)?(?:#(.*))?", link)
    if m:
        owner, repo, kind, rest, anchor = m.groups()
        d.update(owner=owner, repo=repo, type="file" if kind else "repo")
        if kind in ("issues", "pull"):
            parts = rest.split("/")
            d.update(number=int(parts[0]), type="pr" if kind == "pull" else "issue")
            if anchor and anchor.startswith("issuecomment-"):
                d.update(type="comment", comment_id=int(anchor.split("-")[1]))
            if anchor and anchor.startswith("pullrequestreview-"):
                d.update(type="review", review_id=int(anchor.split("-")[1]))
            if kind == "pull" and len(parts) > 2 and parts[1] in ("changes", "commits") and re.fullmatch(r"[0-9a-f]{7,40}", parts[2]):
                d.update(type="commit", sha=parts[2])
        if kind == "commit":
            d.update(type="commit", sha=rest.split("/")[0])
        if kind == "compare":
            d.update(type="compare", compare=rest)
    links[link] = d
print(len(links), "links")
pd.DataFrame(links.values())["type"].value_counts()

Look each one up on GitHub: who made it, when, and how long it is. For an issue or pull request,
also who else commented on it or reviewed it.

In [ ]:
for link, d in links.items():
    base = f"https://api.github.com/repos/{d['owner']}/{d['repo']}"
    d.update(author=None, created_at=None, size=None, participants=[])
    if d["number"]:
        r = github.get(f"{base}/issues/{d['number']}")
        item = r.json() if r.ok else {}
        d.update(author=(item.get("user") or {}).get("login"), created_at=item.get("created_at"), size=len(item.get("body") or ""))
        people, page = [], 1
        while True:
            r = github.get(f"{base}/issues/{d['number']}/comments", params={"per_page": 100, "page": page})
            batch = r.json() if r.ok else []
            if not batch:
                break
            people += [(c.get("user") or {}).get("login") for c in batch]
            page += 1
        if item.get("pull_request"):
            r = github.get(f"{base}/pulls/{d['number']}/reviews", params={"per_page": 100})
            people += [(c.get("user") or {}).get("login") for c in (r.json() if r.ok else [])]
        d["participants"] = sorted({p for p in people if p})
    if d["type"] == "comment":
        r = github.get(f"{base}/issues/comments/{d['comment_id']}")
        c = r.json() if r.ok else {}
        d.update(author=(c.get("user") or {}).get("login"), created_at=c.get("created_at"), size=len(c.get("body") or ""))
    if d["type"] == "review":
        r = github.get(f"{base}/pulls/{d['number']}/reviews/{d['review_id']}")
        c = r.json() if r.ok else {}
        d.update(author=(c.get("user") or {}).get("login"), created_at=c.get("submitted_at"), size=len(c.get("body") or ""))
    if d["type"] == "commit":
        r = github.get(f"{base}/commits/{d['sha']}")
        c = r.json() if r.ok else {}
        d.update(author=(c.get("author") or {}).get("login"), created_at=((c.get("commit") or {}).get("author") or {}).get("date"))
    if d["type"] == "compare":
        r = github.get(f"{base}/compare/{d['compare']}")
        last = ((r.json() if r.ok else {}).get("commits") or [{}])[-1]
        d.update(author=(last.get("author") or {}).get("login"), created_at=((last.get("commit") or {}).get("author") or {}).get("date"))
print(sum(bool(d["author"]) for d in links.values()), "of", len(links), "links have an author on GitHub")

Put what GitHub said next to each submission. Each student's account is the author of most of
their links.

In [ ]:
for field in ("type", "owner", "repo", "author", "created_at", "size", "participants"):
    revs[field] = [links[u][field] if u in links else None for u in revs["url"]]
revs[["type", "owner", "repo"]] = revs[["type", "owner", "repo"]].fillna("")
revs["item"] = [links[u]["comment_id"] or links[u]["review_id"] or links[u]["sha"] or links[u]["number"] or re.sub(r"[#?].*$", "", u).lower() if u in links else None
                for u in revs["url"]]
top = revs.dropna(subset=["author"]).groupby("user_id")["author"].value_counts().reset_index().drop_duplicates("user_id")
revs["account"] = revs["user_id"].map(dict(zip(top["user_id"], top["author"])))
revs["made_week"] = [(datetime.fromisoformat(c.replace("Z", "+00:00")).astimezone(spec.TZ).date() - spec.week_monday(1)).days // 7 + 1
                     if isinstance(c, str) else None for c in revs["created_at"]]
revs["why"] = ""
revs[revs["submitted"]][["week", "name", "type", "author", "account", "made_week"]].head(10)

### Revisions for you to decide

Each cell prints one kind of link that isn't plain, and marks its rows. A row can show up in
more than one cell. Marked rows are left out of the table, so enter their scores in Canvas
yourself.

**Links GitHub doesn't know**: not a GitHub link, or a page with no author (deleted, private, or
mistyped).

In [ ]:
mask = revs["submitted"] & revs["author"].isna()
revs.loc[mask, "why"] += "no author on GitHub; "
print(mask.sum(), "links")
revs[mask][["week", "name", "type", "url"]]

**Someone else's item**: the author isn't the student's account. Either it's the student's second
account, or the student linked another person's issue; `commented` says whether the student's
account is among the item's participants.

In [ ]:
mask = revs["submitted"] & revs["author"].notna() & (revs["author"] != revs["account"])
revs.loc[mask, "why"] += "someone else's item; "
print(mask.sum(), "links")
revs[mask].assign(commented=[acct in (p or []) for acct, p in zip(revs.loc[mask, "account"], revs.loc[mask, "participants"])])[
    ["week", "name", "author", "account", "commented", "url"]]

**Outside the class repositories**: neither a class repository nor a fork on the student's own
account.

In [ ]:
mask = revs["submitted"] & (revs["type"] != "not github") & ~(revs["owner"] + "/" + revs["repo"]).str.lower().isin(CLASS_REPOS) \
       & (revs["owner"].str.lower() != revs["account"].str.lower())
revs.loc[mask, "why"] += "outside the class repositories; "
print(mask.sum(), "links")
revs[mask][["week", "name", "owner", "repo", "url"]]

**A repository or file page**, not an item someone made.

In [ ]:
mask = revs["submitted"] & revs["type"].isin(["repo", "file"])
revs.loc[mask, "why"] += "a repository or file page; "
print(mask.sum(), "links")
revs[mask][["week", "name", "url"]]

**The same item in more than one week.** It earns credit once; `made_week` is the week GitHub
says it was made.

In [ ]:
key = revs["user_id"].astype(str) + " " + (revs["owner"] + "/" + revs["repo"]).str.lower() + " " + revs["item"].astype(str)
mask = revs["submitted"] & revs["item"].notna() & key.duplicated(keep=False)
revs.loc[mask, "why"] += "same item in another week; "
print(mask.sum(), "links")
revs[mask].sort_values(["name", "week"])[["week", "made_week", "name", "type", "url"]]

**Short comments and reviews**, under 80 characters.

In [ ]:
mask = revs["submitted"] & revs["type"].isin(["comment", "review"]) & (revs["size"].fillna(0) < 80)
revs.loc[mask, "why"] += "short; "
print(mask.sum(), "links")
revs[mask][["week", "name", "size", "url"]]

### The plain table

The unmarked rows: 1 for a plain link, 0 for no submission.

In [ ]:
rows = []
for _, s in revs[revs["why"] == ""].iterrows():
    rows.append({"kind": "revision", "week": s["week"], "assignment_id": s["assignment_id"], "user_id": s["user_id"],
                 "name": s["name"], "score": int(s["submitted"]), "rubric": "",
                 "notes": s["type"] if s["submitted"] else "no submission", "url": s["url"]})
revisions = pd.DataFrame(rows, columns=COLUMNS)
path = GRADING_DIR / "revisions-w02-09.csv"
revisions.to_csv(path, index=False)
print(len(revisions), "rows saved to", path, "|", (revs["why"] != "").sum(), "left for you")
revisions.groupby("week")["score"].agg(["count", "sum"])

## Portfolio checkpoint, weeks 11–14

Each student submits the URL of their public project repository, which is graded as it stood at
the deadline. A checkpoint is **plain** when the repository meets both of the week's criteria
(`docs/grading.md`); it earns 2. No submission earns 0. Anything else, including a repository
that meets one criterion, is printed for you to decide.

First, the week's submissions.

In [ ]:
WEEK = 11

a = assignments[(assignments["group"] == "Textbook Revisions") & (assignments["week"] == WEEK)].iloc[0]
url, params, rows = f"{CANVAS}/assignments/{a['assignment_id']}/submissions", {"per_page": 100}, []
while url:
    r = canvas.get(url, params=params)
    r.raise_for_status()
    for s in r.json():
        if s["user_id"] in roster.index:
            rows.append({"user_id": s["user_id"], "name": roster.loc[s["user_id"], "name"], "url": (s.get("url") or "").strip()})
    url, params = r.links.get("next", {}).get("url"), None
port = pd.DataFrame(rows, columns=["user_id", "name", "url"])
port["repo"] = port["url"].str.extract(r"https?://github\.com/([^/]+/[^/#?]+)", expand=False).str.removesuffix(".git")
port["why"] = ""
print(f"{a['name']}, due {a['due_at']}:", (port["url"] != "").sum(), "of", len(port), "submitted")
port.head()

**Links that aren't a GitHub repository**, for you to decide.

In [ ]:
mask = (port["url"] != "") & port["repo"].isna()
port.loc[mask, "why"] += "not a GitHub repository link; "
print(mask.sum(), "links")
port[mask][["name", "url"]]

Clone each repository into `GRADING_DIR/repos` (or fetch it, if it's there from an earlier week),
then find its last commit before the deadline.

In [ ]:
port["commit"] = ""
for i, s in port[port["repo"].notna()].iterrows():
    repo = GRADING_DIR / "repos" / s["repo"].replace("/", "__").lower()
    if repo.exists():
        done = subprocess.run(["git", "-C", repo, "fetch", "-q", "origin"])
    else:
        done = subprocess.run(["git", "clone", "-q", "--filter=blob:none", f"https://github.com/{s['repo']}", repo])
    if done.returncode == 0:
        port.at[i, "commit"] = subprocess.run(["git", "-C", repo, "rev-list", "-1", f"--before={a['due_at']}", "origin/HEAD"],
                                              capture_output=True, text=True).stdout.strip()
print((port["commit"] != "").sum(), "repositories with a commit before the deadline")
port[port["repo"].notna()][["name", "repo", "commit"]].head()

**Repositories with nothing to grade**: they couldn't be cloned (private, renamed, or deleted),
or had no commit before the deadline.

In [ ]:
mask = port["repo"].notna() & (port["commit"] == "")
port.loc[mask, "why"] += "couldn't clone, or nothing pushed by the deadline; "
print(mask.sum(), "repositories")
port[mask][["name", "url"]]

Check out each commit and take an inventory: the README's headings and the length of the sections
the criteria name, the notebooks and how many of their code cells have output, and the data
files, figures, and report.

In [ ]:
facts = {}
for i, s in port[port["commit"] != ""].iterrows():
    repo = GRADING_DIR / "repos" / s["repo"].replace("/", "__").lower()
    subprocess.run(["git", "-C", repo, "checkout", "-q", "--detach", s["commit"]])
    files = [p for p in repo.rglob("*") if p.is_file() and ".git" not in p.parts]
    readme = next((p for p in files if p.name.lower() == "readme.md" and p.parent == repo), None)
    text = readme.read_text(errors="replace") if readme else ""
    f = {"files": len(files), "names": {p.name.lower() for p in files}, "readme_words": len(text.split()),
         "headings": " ".join(h.lower() for h in re.findall(r"^#+\s*(.+)$", text, re.M)),
         "notebooks": [p for p in files if p.suffix == ".ipynb"],
         "code_files": [p for p in files if p.suffix in (".py", ".ipynb")],
         "data_dir": (repo / "data").is_dir(),
         "data_files": [p for p in files if "data" in [x.lower() for x in p.relative_to(repo).parts[:-1]]],
         "figures": [p for p in files if p.suffix.lower() in (".png", ".jpg", ".jpeg", ".svg") or (p.suffix.lower() == ".pdf" and p.name.lower() != "report.pdf")],
         "report": next((p for p in files if p.name.lower() in ("report.md", "report.pdf")), None),
         "mentions_report": "report" in text.lower(), "code_cells": 0, "cells_with_output": 0}
    for name in ("data", "method", "finding", "reproduc", "how to"):
        found = re.findall(rf"^#+\s*{name}[^\n]*\n(.*?)(?=^#|\Z)", text, re.M | re.S | re.I)
        f[f"{name}_words"] = max([len(x.split()) for x in found] + [0])
    for nb in f["notebooks"]:
        try:
            nb_cells = json.loads(nb.read_text(errors="replace")).get("cells", [])
        except ValueError:
            nb_cells = []
        for c in nb_cells:
            if c.get("cell_type") == "code" and "".join(c.get("source", [])).strip():
                f["code_cells"] += 1
                f["cells_with_output"] += bool(c.get("outputs"))
    f["report_words"] = len(f["report"].read_text(errors="replace").split()) if f["report"] is not None and f["report"].suffix == ".md" else None
    facts[i] = f
print(len(facts), "repositories checked")
pd.DataFrame({i: {k: (len(v) if isinstance(v, (list, set)) else v) for k, v in f.items() if k != "headings"} for i, f in facts.items()}).T.head()

The week's two criteria, from the inventory. They look for the files and README sections the week
asks for, so they're a first pass: open the repository when one fails.

| Week | Criterion 1 | Criterion 2 |
|---|---|---|
| 11 | a README of 40+ words with three of the four sections | `proposal.md`, a notebook, a report or its placeholder, and `data/` |
| 12 | collection code that sets a User-Agent or headers | data files, and a Data section of 40+ words |
| 13 | code cells, at least half of them with output | a figure, and Methods and Findings sections of 20+ words each |
| 14 | `report.md` of 1,000+ words, or `report.pdf` | all four required files, and 20+ words on reproducing it |

In [ ]:
port["c1"], port["c2"] = False, False
for i, f in facts.items():
    if WEEK == 11:
        c1 = f["readme_words"] >= 40 and sum(h in f["headings"] for h in ("data", "method", "finding", "reproduc")) >= 3
        c2 = "proposal.md" in f["names"] and bool(f["notebooks"]) and (f["report"] is not None or f["mentions_report"]) and f["data_dir"]
    if WEEK == 12:
        c1 = any(re.search(r"user-agent|headers\s*=", p.read_text(errors="replace"), re.I) for p in f["code_files"] if p.stat().st_size < 5_000_000)
        c2 = bool(f["data_files"]) and f["data_words"] >= 40
    if WEEK == 13:
        c1 = f["code_cells"] > 0 and f["cells_with_output"] >= max(1, f["code_cells"] // 2)
        c2 = bool(f["figures"]) and f["method_words"] >= 20 and f["finding_words"] >= 20
    if WEEK == 14:
        c1 = f["report"] is not None and (f["report_words"] is None or f["report_words"] >= 1000)
        c2 = {"readme.md", "proposal.md"} <= f["names"] and bool(f["notebooks"]) and f["report"] is not None and f["reproduc_words"] + f["how to_words"] >= 20
    port.at[i, "c1"], port.at[i, "c2"] = c1, c2
port[port["commit"] != ""][["c1", "c2"]].value_counts()

**Repositories missing a component**: one criterion or both not met. The inventory shows what
was found.

In [ ]:
mask = (port["commit"] != "") & ~(port["c1"] & port["c2"])
port.loc[mask, "why"] += "missing a component; "
print(mask.sum(), "repositories")
pd.DataFrame([{"name": port.at[i, "name"], "c1": port.at[i, "c1"], "c2": port.at[i, "c2"], "headings": facts[i]["headings"][:80],
               "notebooks": len(facts[i]["notebooks"]), "data files": len(facts[i]["data_files"]), "figures": len(facts[i]["figures"]),
               "report": facts[i]["report"].name if facts[i]["report"] is not None else "", "url": port.at[i, "url"]}
              for i in port[mask].index])

### The plain table

2 for a repository meeting both criteria, 0 for no submission.

In [ ]:
rows = []
for i, s in port[port["why"] == ""].iterrows():
    done = s["url"] != ""
    rows.append({"kind": "portfolio", "week": WEEK, "assignment_id": a["assignment_id"], "user_id": s["user_id"],
                 "name": s["name"], "score": 2 if done else 0, "rubric": "[1, 1]" if done else "",
                 "notes": f"commit {s['commit'][:7]}" if done else "no submission", "url": s["url"]})
portfolio = pd.DataFrame(rows, columns=COLUMNS)
path = GRADING_DIR / f"portfolio-w{WEEK:02d}.csv"
portfolio.to_csv(path, index=False)
print(len(portfolio), "rows saved to", path, "|", (port["why"] != "").sum(), "left for you")
portfolio["score"].value_counts()

## Notebook Labs

Students upload the chapter's companion notebook as HTML, and only its Recommended Exercises are
graded. A lab is **plain** when it was submitted on time as one readable HTML file, with the
student's own code in every step, every code cell run, and an interpretation of 50 or more new
words. It earns 10, all five rubric criteria Met. No submission earns 0. Any other lab is printed
for you to decide, including a late one, a skipped optional step, or a blank scaffold.

First, download every upload. Each is saved as `.html.txt`, so it's never opened as a web page.

In [ ]:
WEEK = 11

a = assignments[(assignments["group"] == "Notebook Labs") & (assignments["week"] == WEEK)].iloc[0]
lab_dir = GRADING_DIR / "labs" / f"week-{WEEK:02d}"
lab_dir.mkdir(parents=True, exist_ok=True)
url, params, rows = f"{CANVAS}/assignments/{a['assignment_id']}/submissions", {"per_page": 100}, []
while url:
    r = canvas.get(url, params=params)
    r.raise_for_status()
    for s in r.json():
        if s["user_id"] not in roster.index:
            continue
        files = []
        for i, att in enumerate(s.get("attachments") or []):
            p = lab_dir / f"{s['user_id']}_{i}.html.txt"
            p.write_bytes(canvas.get(att["url"]).content)
            files.append(p)
        rows.append({"user_id": s["user_id"], "name": roster.loc[s["user_id"], "name"], "submitted_at": s.get("submitted_at"), "files": files})
    url, params = r.links.get("next", {}).get("url"), None
lab = pd.DataFrame(rows, columns=["user_id", "name", "submitted_at", "files"])
lab["why"] = ""
print(f"{a['name']}, due {a['due_at']}:", (lab["files"].str.len() > 0).sum(), "of", len(lab), "submitted")
lab["files"].str.len().value_counts()

Get the companion notebook as it stood before the due date: the scaffold the students filled in.
Some versions in the book were broken JSON, so step back to the most recent one that parses.

In [ ]:
slug = spec.week_info(WEEK)[4]
r = github.get(f"https://api.github.com/repos/cuinfoscience/Web-Data-Science-Book/commits",
               params={"path": f"notebooks/{slug}.ipynb", "until": a["due_at"], "per_page": 10})
scaffold_nb, scaffold_sha = None, None
for commit in (r.json() if r.ok else []):
    raw = requests.get(f"https://raw.githubusercontent.com/cuinfoscience/Web-Data-Science-Book/{commit['sha']}/notebooks/{slug}.ipynb")
    try:
        scaffold_nb, scaffold_sha = raw.json(), commit["sha"]
        break
    except ValueError:
        print(commit["sha"][:7], "is broken JSON; trying the one before")
print(f"notebooks/{slug}.ipynb from", scaffold_sha[:7] if scaffold_sha else "nowhere: check the book")

Split the scaffold's Recommended Exercises into steps, the way the students' copies will be split.
Each step keeps its code lines and its words, so that later the scaffold's can be subtracted from
each student's, leaving the student's own work. The last step is the interpretation.

In [ ]:
STEP = re.compile(r"(?:^|\n)\s*(?:#+\s*)?(?:\*\*)?Step\s*(\d+)\s*(?:—|–|-|:|\.|\*\*|¶|$)", re.I | re.M)
START = re.compile(r"^\s*#*\s*Recommended Exercises", re.M)
END = re.compile(r"^\s*#*\s*(Additional Exercises|Social History|Public Interest|Common Issues|Key Takeaways|Further Reading|Missing Manual Reference)", re.I)

scaffold_code, scaffold_words, step, inside = defaultdict(Counter), defaultdict(Counter), 0, False
for c in scaffold_nb["cells"]:
    src = "".join(c["source"])
    if c["cell_type"] == "markdown" and START.search(src):
        inside = True
    elif inside and c["cell_type"] == "markdown" and END.search(re.sub(r"[*`_#>]", "", src)):
        break
    if not inside:
        continue
    found = [int(x) for x in STEP.findall(re.sub(r"[`_>]", "", src)) if 0 < int(x) < 12]
    if found:
        step = max(found)
    if c["cell_type"] == "code":
        scaffold_code[step].update(l.strip() for l in src.splitlines() if l.strip() and not l.strip().startswith("#"))
    scaffold_words[step].update(w.lower() for w in re.findall(r"[A-Za-z']{2,}", re.sub(r"[*`_#>]", "", src)))
last_step = max(scaffold_words) if scaffold_words else 0
code_steps = list(range(1, last_step))
print("Steps:", last_step, "| code steps:", code_steps, "| interpretation: step", last_step)
pd.DataFrame({"code lines": {k: sum(v.values()) for k, v in scaffold_code.items()},
              "words": {k: sum(v.values()) for k, v in scaffold_words.items()}}).fillna(0).astype(int)

Read each upload into cells. A code cell keeps its source, whether it ran (its prompt has a
number), how many outputs it has, and whether one is a traceback.

In [ ]:
uploads = {}
for files in lab["files"]:
    for p in files:
        soup = BeautifulSoup(p.read_text(errors="replace"), "lxml")
        cells = []
        for c in soup.select("div.jp-Cell") or soup.select("div.cell"):
            classes = c.get("class", [])
            if "jp-MarkdownCell" in classes or "text_cell" in classes:
                cells.append({"md": True, "text": c.get_text("\n", strip=True)})
            elif "jp-CodeCell" in classes or "code_cell" in classes:
                editor = c.select_one(".jp-InputArea-editor, .input_area")
                outputs = c.select(".jp-OutputArea-output, .output_area")
                prompt = c.select_one(".jp-InputPrompt, .input_prompt")
                out_text = "\n".join(o.get_text("\n", strip=True) for o in outputs)
                cells.append({"md": False, "text": editor.get_text() if editor else "", "outputs": len(outputs),
                              "ran": bool(prompt and re.search(r"\d", prompt.get_text())),
                              "traceback": bool(re.search(r"Traceback \(most recent call last\)|^\w+(Error|Exception):", out_text, re.M))})
        uploads[p] = cells
print(len(uploads), "files read")
pd.Series({p.name: len(c) for p, c in uploads.items()}, name="cells").head()

Check each file against the scaffold:
1. Find its Recommended Exercises section, or failing that its Step 1.
2. Give each cell in the section to a step.
3. Count the student's own code lines in each code step, and the new words in the interpretation
   step. An interpretation can be in the prompt cell, a comment, a Markdown cell, or a `print()`.
4. Count code cells with the student's code, how many are unrun, and their outputs.

In [ ]:
rows = []
for p, cells in uploads.items():
    start = next((i for i, c in enumerate(cells) if c["md"] and START.search(c["text"])), None)
    if start is None:
        start = next((i for i, c in enumerate(cells) if re.search(r"Step\s*1\b", c["text"])), None)
    check = {"file": p, "cells": len(cells), "found": start is not None, "attempted": [], "interp": 0,
             "code_cells": 0, "unrun": 0, "outputs": 0, "traceback": False, "section_text": "", "markdown_text": ""}
    if start is not None:
        section = []
        for c in cells[start:]:
            if section and c["md"] and END.search(c["text"]):
                break
            section.append(c)
        code, words, step = defaultdict(Counter), defaultdict(Counter), 0
        for c in section:
            found = [int(x) for x in STEP.findall(c["text"]) if 0 < int(x) < 12]
            if found:
                step = max(found)
            if not c["md"]:
                code[step].update(l.strip() for l in c["text"].splitlines() if l.strip() and not l.strip().startswith("#"))
                if any(l.strip() and not l.strip().startswith("#") for l in c["text"].splitlines()):
                    check["code_cells"] += 1
                    check["unrun"] += not c["ran"]
                    check["outputs"] += c["outputs"]
                    check["traceback"] |= c["traceback"]
            words[step].update(w.lower() for w in re.findall(r"[A-Za-z']{2,}", c["text"]))
        check["attempted"] = [k for k in code_steps if sum((code[k] - scaffold_code[k]).values()) > 0]
        check["interp"] = sum((words[last_step] - scaffold_words[last_step]).values())
        check["section_text"] = "\n".join(c["text"] for c in section)
        check["markdown_text"] = "\n".join(c["text"] for c in section if c["md"])
    rows.append(check)
checks = pd.DataFrame(rows)
print(len(checks), "files checked;", (~checks["found"]).sum(), "with no Recommended Exercises or Step 1")
checks.drop(columns=["section_text", "markdown_text"]).head()

An interpretation written under its own heading, such as "Final interpretation", sits after the
last step label. Count its new words for files whose interpretation step came out under three.

In [ ]:
for i, ch in checks[checks["found"] & (checks["interp"] < 3)].iterrows():
    extra = Counter(w.lower() for c in uploads[ch["file"]] if c["md"] and re.match(r"\s*(final\s+)?interpret", c["text"], re.I)
                    for w in re.findall(r"[A-Za-z']{2,}", c["text"]))
    checks.at[i, "interp"] = max(ch["interp"], sum((extra - scaffold_words[last_step]).values()))
print((checks["interp"] < 3).sum(), "files still under three words")

A student who uploaded two files is graded on the fuller one: more steps attempted, then more
interpretation. Join each student's to the submissions, and work out how late each one was.

In [ ]:
checks["steps_done"] = checks["attempted"].str.len()
checks["user_id"] = [int(p.name.split("_")[0]) for p in checks["file"]]
fullest = checks.sort_values(["steps_done", "interp"], ascending=False).drop_duplicates("user_id").set_index("user_id")
lab = lab.join(fullest.drop(columns=["file"]), on="user_id")
lab["hours_late"] = [(datetime.fromisoformat(t.replace("Z", "+00:00")) - datetime.fromisoformat(a["due_at"].replace("Z", "+00:00"))).total_seconds() / 3600
                     if isinstance(t, str) else None for t in lab["submitted_at"]]
lab["submitted"] = lab["files"].str.len() > 0
lab[lab["submitted"]][["name", "steps_done", "interp", "code_cells", "unrun", "outputs", "traceback", "hours_late"]].head(10)

### Labs for you to decide

Each cell prints one kind of lab that isn't plain, and marks its rows. A lab can show up in more
than one cell. Marked labs are left out of the table, so grade them in Canvas yourself, with the
rules in `docs/grading.md`.

**Files the checks can't read**: not a Jupyter HTML export, or no Recommended Exercises section
and no Step 1.

In [ ]:
mask = lab["submitted"] & (lab["found"] != True)
lab.loc[mask, "why"] += "unreadable; "
print(mask.sum(), "labs")
lab[mask].assign(file=[", ".join(p.name for p in f) for f in lab.loc[mask, "files"]])[["name", "file", "cells"]]

**More than one file**. The fuller file was checked; make sure it's the one to grade.

In [ ]:
mask = lab["files"].str.len() > 1
lab.loc[mask, "why"] += "more than one file; "
print(mask.sum(), "labs")
lab[mask].assign(files=[", ".join(p.name for p in f) for f in lab.loc[mask, "files"]])[["name", "files", "steps_done", "interp"]]

**The blank scaffold**: no code of the student's own, and almost no interpretation.

In [ ]:
mask = lab["submitted"] & (lab["found"] == True) & (lab["steps_done"] == 0) & (lab["interp"] < 10)
lab.loc[mask, "why"] += "blank scaffold; "
print(mask.sum(), "labs")
lab[mask][["name", "steps_done", "interp"]]

**Steps with none of the student's code**, including any step you told students to skip.

In [ ]:
mask = lab["submitted"] & (lab["found"] == True) & (lab["steps_done"] < len(code_steps))
lab.loc[mask, "why"] += "steps missing; "
print(mask.sum(), "labs")
lab[mask].assign(missing=[[k for k in code_steps if k not in done] for done in lab.loc[mask, "attempted"]])[["name", "missing", "interp"]]

**Code not run**: cells with the student's code and no execution count, or no code cells at all.

In [ ]:
mask = lab["submitted"] & (lab["found"] == True) & ((lab["unrun"] > 0) | (lab["code_cells"] == 0))
lab.loc[mask, "why"] += "code not run; "
print(mask.sum(), "labs")
lab[mask][["name", "code_cells", "unrun", "outputs"]]

**A thin interpretation**: under 50 new words.

In [ ]:
mask = lab["submitted"] & (lab["found"] == True) & (lab["interp"] < 50)
lab.loc[mask, "why"] += "thin interpretation; "
print(mask.sum(), "labs")
lab[mask][["name", "interp"]]

**An interpretation that cites numbers with no output to back them**, and **phrases that look
AI-generated**: notes addressed to a reader, or a placeholder's wording left in.

In [ ]:
lab["numbers"] = [bool(re.search(r"\d{2,}", t)) and o == 0 if isinstance(t, str) else False for t, o in zip(lab["markdown_text"], lab["outputs"])]
lab["ai_signs"] = [bool(re.search(r"revise it to reflect your own|no measurements have been invented|example interpretation", t, re.I)) if isinstance(t, str) else False
                   for t in lab["section_text"]]
mask = lab["submitted"] & (lab["numbers"] | lab["ai_signs"])
lab.loc[mask, "why"] += "numbers with no output, or AI phrasing; "
print(mask.sum(), "labs")
lab[mask][["name", "numbers", "ai_signs", "outputs"]]

**Submitted after the due time.** `hours_late` is from the due time; the grace period and the
penalty are in `docs/grading.md`.

In [ ]:
mask = lab["submitted"] & (lab["hours_late"] > 0)
lab.loc[mask, "why"] += "late; "
print(mask.sum(), "labs")
lab[mask][["name", "submitted_at", "hours_late"]].round(1)

### The plain table

10 for a plain lab, with every rubric criterion Met when the assignment has a rubric; 0 for no
submission. Open a few of the plain labs' HTML to spot-check criterion 5, whether each step does
its task.

In [ ]:
rows = []
for _, s in lab[lab["why"] == ""].iterrows():
    rows.append({"kind": "lab", "week": WEEK, "assignment_id": a["assignment_id"], "user_id": s["user_id"], "name": s["name"],
                 "score": 10 if s["submitted"] else 0, "rubric": "[2, 2, 2, 2, 2]" if s["submitted"] and a["has_rubric"] else "",
                 "notes": f"{int(s['interp'])} words of interpretation" if s["submitted"] else "no submission", "url": ""})
labs = pd.DataFrame(rows, columns=COLUMNS)
path = GRADING_DIR / f"labs-w{WEEK:02d}.csv"
labs.to_csv(path, index=False)
print(len(labs), "rows saved to", path, "|", (lab["why"] != "").sum(), "left for you")
labs["score"].value_counts()

## Post a table

Set `TABLE` to a CSV saved above. Each row's score is posted to Canvas, through the rubric when the
row has rubric points, with no comment. Nothing is sent until `APPLY = True` and `CONFIRM` matches
the number of rows.

In [ ]:
TABLE = "labs-w11"
CONFIRM = ""           # type "post N", where N is the number of rows printed below

table = pd.read_csv(GRADING_DIR / f"{TABLE}.csv", dtype=str, keep_default_na=False)
print(len(table), "rows to post from", TABLE)
table["score"].value_counts()

In [ ]:
if not APPLY:
    print("Dry run: set APPLY = True in Setup to post.")
elif CONFIRM != f"post {len(table)}":
    print(f'Set CONFIRM = "post {len(table)}" to post these rows.')
else:
    criteria = {}
    for _, row in table.iterrows():
        data = {"submission[posted_grade]": row["score"]}
        if row["rubric"]:
            if row["assignment_id"] not in criteria:
                criteria[row["assignment_id"]] = [c["id"] for c in canvas.get(f"{CANVAS}/assignments/{row['assignment_id']}").json().get("rubric") or []]
            points = json.loads(row["rubric"])
            if len(points) != len(criteria[row["assignment_id"]]):
                raise ValueError(f"{row['name']}: the rubric has {len(criteria[row['assignment_id']])} criteria and the row has {len(points)}")
            data = {f"rubric_assessment[{c}][points]": p for c, p in zip(criteria[row["assignment_id"]], points)}
        canvas.put(f"{CANVAS}/assignments/{row['assignment_id']}/submissions/{row['user_id']}", data=data).raise_for_status()
    print("Posted", len(table), "rows")

Read the table back from Canvas and list any row whose score doesn't match. Run this any time to
check a table you posted earlier.

In [ ]:
live = {}
for assignment_id in sorted(set(table["assignment_id"])):
    url, params = f"{CANVAS}/assignments/{assignment_id}/submissions", {"per_page": 100}
    while url:
        r = canvas.get(url, params=params)
        r.raise_for_status()
        for s in r.json():
            live[(str(assignment_id), str(s["user_id"]))] = s.get("score")
        url, params = r.links.get("next", {}).get("url"), None
table["canvas"] = [live.get((row["assignment_id"], row["user_id"])) for _, row in table.iterrows()]
mismatches = table[[c is None or float(c) != float(s) for c, s in zip(table["canvas"], table["score"])]]
print(len(mismatches), "of", len(table), "rows don't match Canvas")
mismatches[["name", "week", "score", "canvas"]]